# ⚡ Predicting Electric Vehicle Purchases — Starter (Kaggle S6E9)

> **Competition:** [Playground Series — Season 6, Episode 9: Predicting Electric Vehicle Purchases](https://www.kaggle.com/competitions/playground-series-s6e9)
> **Task:** binary classification — predict the probability a consumer buys an EV (`Will_Buy_EV`)
> **Metric:** ROC-AUC (higher is better)

## What this notebook does

A full, leak-safe pipeline that reproduces the top public-Leaderboard recipe
(~**0.9444** public AUC) in a single notebook:

1. **EDA** — class balance, per-feature signals, and the *synthetic-data artifact*
   hiding in `Annual_Income_USD` (its decimal digits are predictive!).
2. **Feature engineering**
   - 6 income *digit* features (ones → hundred-thousands)
   - fold-local, leak-safe **frequency encoding** of income
3. **Three 5-fold CV models** (StratifiedKFold, seed 42)
   - LightGBM (1000 trees, lr 0.05)
   - CatBoost (700 iters, depth 7, lr 0.08, early stopping 50)
   - strong LightGBM (1500 trees, lr 0.03, `max_bin=2047`)
4. **Probability ensemble** with weights 0.20 / 0.20 / 0.60 → `submission.csv`

## Expected results (reference solution, full 668k-row data)

| Stage | OOF AUC |
|---|---|
| LightGBM, raw features (baseline) | ≈ 0.9417 |
| LightGBM + digits + income frequency | ≈ 0.9442 |
| Final 3-model ensemble | **≈ 0.9444** (public LB 0.94439) |

## How to run

- **On Kaggle:** upload this notebook to the competition, *Save Version* →
  *Run* (Internet off, CPU 9h). Data is auto-mounted at
  `/kaggle/input/playground-series-s6e9/`.
- **Locally:** put `train.csv`, `test.csv`, `sample_submission.csv` next to the
  notebook or in `./data/`.
- **Quick check:** set `FAST = True` in the setup cell to run on a 100k-row subsample.
- Full-data runtime is roughly **30–45 min on Kaggle CPU**.

## 0 · Setup

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.grid": True, "grid.alpha": 0.3})
warnings.filterwarnings("ignore")

SEED        = 42     # global random seed (matches the reference solution)
N_FOLDS     = 5      # cross-validation folds
FAST        = False  # True  -> smoke test on a 100k-row subsample
                       # False -> full 668k rows (best results)
SUBMISSION_PATH = "submission.csv"

## 1 · Load data

Data lives in the Kaggle input directory when run on Kaggle; locally it is
picked up from the notebook folder or `./data/`.

In [ ]:
def find_data_dir() -> Path:
    for d in (Path("/kaggle/input/playground-series-s6e9"), Path.cwd(), Path("data")):
        if (d / "train.csv").is_file() and (d / "test.csv").is_file():
            return d
    raise FileNotFoundError(
        "train.csv / test.csv not found. Run this on Kaggle (competition "
        "'playground-series-s6e9') or drop the CSVs next to the notebook / in ./data."
    )


DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR)

train  = pd.read_csv(DATA_DIR / "train.csv")
test   = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

TARGET = "Will_Buy_EV"
ID_COL = "id"

# Accept both "Yes"/"No" and 0/1 labels
train[TARGET] = train[TARGET].astype(str).str.strip().str.lower()
y = (train[TARGET] == "yes").astype(int)
train[TARGET] = y

if FAST:
    train, _ = train_test_split(train, train_size=100_000, stratify=y, random_state=SEED)
    y = y.loc[train.index]
    print("FAST mode: training on", len(train), "rows")

print(f"train: {train.shape}  |  test: {test.shape}")
print(f"P(buy) = {y.mean():.4f}  (imbalanced — expect ~0.175)")
train.head()

## 2 · Dataset overview

**13 features**, no missing values, clean tabular data:

| Feature | Type | Meaning |
|---|---|---|
| `Age` | int | customer age (18–85) |
| `Annual_Income_USD` | float | yearly income |
| `Daily_Commute_km` | float | daily commute distance |
| `Number_of_Cars_Owned` | int 0–5 | household fleet size |
| `Charging_Stations_Near_Home` | int 0–15 | chargers near home |
| `Charging_Stations_Near_Work` | int 0–15 | chargers near work |
| `Environmental_Concern_Level` | float 1–5 | eco concern |
| `Gender`, `City_Type`, `Current_Car_Type` | cat | demographics / vehicle |
| `Home_Charging_Possible`, `Subsidy_Available` | Yes/No | infrastructure / incentive |
| `Range_Anxiety_Level` | Low/Med/High | battery-range concern |

In [ ]:
feat_cols = [c for c in train.columns if c not in (ID_COL, TARGET)]

overview = pd.DataFrame(
    {
        "dtype": train[feat_cols].dtypes.astype(str),
        "missing %": (train[feat_cols].isna().mean() * 100).round(3),
        "nunique": train[feat_cols].nunique(),
    }
)
overview.loc["Will_Buy_EV (target)"] = ["int64", 0.0, 2]
overview

## 3 · EDA — what drives the decision

Two things to look for:

1. **Which features separate buyers from non-buyers?** Charging access and
   eco concern are the loudest signals; `Gender` is almost useless (trees will
   ignore it — that's fine).
2. **The income artifact** — `Annual_Income_USD` is synthetic: its exact
   values cluster, and even the *trailing digits* correlate with the target.
   Trees can't exploit digit structure from the raw continuous value alone,
   so we engineer it explicitly in §5.

In [ ]:
cat_cols = [c for c in feat_cols if train[c].dtype == object]
num_cols = [c for c in feat_cols if c not in cat_cols]
print("categorical:", cat_cols)
print("numeric:    ", num_cols)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), cat_cols):
    rates = y.groupby(train[col]).mean().sort_values(ascending=False)
    rates.plot(kind="bar", ax=ax, color="#2a9d8f")
    ax.set_title(f"P(buy) by {col}", fontsize=10)
    ax.set_ylabel("P(buy)")
    for i, v in enumerate(rates):
        ax.text(i, v + 0.008, f"{v:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
inc = train["Annual_Income_USD"]

axes[0].hist(inc, bins=250, color="#264653")
axes[0].set_title("Annual_Income_USD — note the clusters, not a smooth distribution")
axes[0].set_xlim(0, inc.max() * 1.02)

decile = pd.qcut(inc, 10, labels=False, duplicates="drop")
y.groupby(decile).mean().plot(kind="bar", ax=axes[1], color="#e76f51")
axes[1].set_title("P(buy) by income decile")
axes[1].set_xlabel("income decile (0 = lowest)")
plt.tight_layout()
plt.show()

print("\nMost common exact income values (clustering at round numbers):")
print(inc.value_counts().head(10))

In [ ]:
# The "aha" moment: target rate by the income *ones digit* —
# a deterministic artifact of how the synthetic data was generated.
ones = train["Annual_Income_USD"].astype(int) % 10
rate = y.groupby(ones).mean()
print("P(buy) by income ones-digit:")
print(rate.map("{:.3f}".format).to_string())

plt.figure(figsize=(7, 4))
rate.plot(kind="bar", color="#264653")
plt.title("P(buy) by income ones-digit (synthetic artifact — real signal)")
plt.ylabel("P(buy)")
plt.xticks(rotation=0)
plt.show()

## 4 · Baseline — LightGBM on raw features

A plain LightGBM (1000 trees, lr 0.05) with 5-fold stratified CV gives the
**≈ 0.9417** reference point. Everything below improves on it.

The helpers defined here (`align_categories`, `cv_lightgbm`) are reused by
all three models.

In [ ]:
import lightgbm as lgb

LGB_BASE = dict(
    n_estimators=1000, learning_rate=0.05, num_leaves=31, max_depth=-1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)
# NOTE: `subsample` has no effect without `subsample_freq` — left at the
# default 0 exactly like the frozen winning recipe.

def cat_columns(df):
    '''Recognize object / pandas-string / category columns.'''
    return [
        c for c in df.columns
        if pd.api.types.is_object_dtype(df[c].dtype)
        or pd.api.types.is_string_dtype(df[c].dtype)
        or isinstance(df[c].dtype, pd.CategoricalDtype)
    ]

def align_categories(*frames):
    '''Copy frames and align LightGBM category codes to one shared vocabulary
    (first-appearance order over train+valid+test). Keeps validation and test
    rows free of "unseen category" problems inside each fold.'''
    out = tuple(f.copy() for f in frames)
    for col in cat_columns(out[0]):
        cats = pd.concat([f[col].astype(object) for f in out]).dropna().unique()
        for f in out:
            f[col] = pd.Categorical(f[col], categories=cats)
    return out

def cv_lightgbm(params, X, X_test, y_, n_splits=N_FOLDS, seed=SEED, freq=False, tag=""):
    '''Stratified K-fold CV for LightGBM.

    If `freq`, the income frequency column is fitted on the *training fold
    only* and applied to validation/test (leak-safe).
    '''
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y_))
    te  = np.zeros(len(X_test))
    scores, t0 = [], time.time()
    for tr_idx, va_idx in skf.split(X, y_):
        f_tr, f_va, f_te = X.iloc[tr_idx], X.iloc[va_idx], X_test
        if freq:
            freq_map = f_tr["Annual_Income_USD"].value_counts(normalize=True, dropna=False)
            for f in (f_tr, f_va, f_te):
                f["Income_freq"] = f["Annual_Income_USD"].map(freq_map).fillna(0.0).astype(float)
        f_tr, f_va, f_te = align_categories(f_tr, f_va, f_te)
        model = lgb.LGBMClassifier(**params)
        model.fit(f_tr, y_.iloc[tr_idx], categorical_feature=cat_columns(f_tr))
        oof[va_idx] = model.predict_proba(f_va)[:, 1]
        te += model.predict_proba(f_te)[:, 1] / n_splits
        scores.append(roc_auc_score(y_.iloc[va_idx], oof[va_idx]))
    auc = roc_auc_score(y_, oof)
    print(f"{tag} fold AUCs: " + "  ".join(f"{s:.5f}" for s in scores))
    print(f"{tag} pooled OOF AUC: {auc:.5f}   ({time.time() - t0:.0f}s)")
    return oof, te

X_raw   = train.drop(columns=[ID_COL, TARGET]).copy()
X_t_raw = test.drop(columns=[ID_COL]).copy()

oof_base, te_base = cv_lightgbm(LGB_BASE, X_raw, X_t_raw, y, freq=False, tag="[LGB raw]")

## 5 · Feature engineering (the key to the top score)

Two additions, both centered on `Annual_Income_USD`:

1. **Income digits** — the six decimal digits (ones … hundred-thousands).
   The synthetic generator left digit-level footprints; individually the
   *hundreds* digit is surprisingly strong. Cost: 6 extra columns.
2. **Fold-local income frequency** — normalized value-count of income, fitted
   **inside each training fold only**, then mapped onto validation and test
   rows (unknown values → 0). This is the leak-safe way to exploit the income
   *clusters* visible in the EDA.

Ablations from the reference solution (LightGBM, pooled 5-fold CV):

| Feature set | OOF AUC |
|---|---|
| raw features | ≈ 0.9417 |
| + 6 income digits | ≈ 0.9438 |
| + fold-local income frequency | **≈ 0.9442** |

In [ ]:
INCOME_COL = "Annual_Income_USD"
INCOME_DIGITS = {
    "Income_ones": 1,
    "Income_tens": 10,
    "Income_hundreds": 100,
    "Income_thousands": 1_000,
    "Income_ten_thousands": 10_000,
    "Income_hundred_thousands": 100_000,
}

def add_income_digits(df: pd.DataFrame) -> pd.DataFrame:
    '''Append the six deterministic decimal digits of Annual_Income_USD.'''
    out = df.copy()
    inc = pd.to_numeric(out[INCOME_COL], errors="raise").round().astype("int64")
    for name, div in INCOME_DIGITS.items():
        out[name] = (inc // div) % 10
    return out

X      = add_income_digits(X_raw)
X_test = add_income_digits(X_t_raw)

print("engineered columns:", [c for c in X.columns if c.startswith("Income_")])
print(X[[INCOME_COL] + list(INCOME_DIGITS)].head())

## 6 · Model 1 — LightGBM V4 (digits + income frequency)

Same architecture as the baseline, now with the engineered features.
Expected OOF AUC **≈ 0.9442** — the +0.0025 comes almost entirely from the
income features.

In [ ]:
oof_lgb, te_lgb = cv_lightgbm(LGB_BASE, X, X_test, y, freq=True, tag="[LGB v4 ]")

## 7 · Model 2 — CatBoost

CatBoost handles categories natively (ordered target statistics) and is the
most different model in the pool — that diversity is what makes the ensemble
worth it. Early stopping (50 rounds) on the validation fold.
Expected OOF AUC **≈ 0.944**.

In [ ]:
from catboost import CatBoostClassifier

CAT_PARAMS = dict(
    iterations=700, depth=7, learning_rate=0.08,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=SEED, verbose=False, thread_count=-1, allow_writing_files=False,
)

def prepare_catboost(df: pd.DataFrame) -> pd.DataFrame:
    '''CatBoost-friendly copy: categoricals as str, NaN -> "__MISSING__".'''
    out = df.copy()
    for col in cat_columns(out):
        out[col] = out[col].astype(object).fillna("__MISSING__").astype(str)
    return out

def cv_catboost(params, X_, X_test_, y_, n_splits=N_FOLDS, seed=SEED):
    '''Stratified K-fold CV for CatBoost, with the same leak-safe
    fold-local income frequency as the LightGBM models.'''
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y_))
    te  = np.zeros(len(X_test_))
    scores, t0 = [], time.time()
    for tr_idx, va_idx in skf.split(X_, y_):
        freq_map = X_.iloc[tr_idx][INCOME_COL].value_counts(normalize=True, dropna=False)
        f_tr = X_.iloc[tr_idx].copy()
        f_va = X_.iloc[va_idx].copy()
        f_te = X_test_.copy()
        for f in (f_tr, f_va, f_te):
            f["Income_freq"] = f[INCOME_COL].map(freq_map).fillna(0.0).astype(float)
        model = CatBoostClassifier(**params)
        model.fit(
            prepare_catboost(f_tr), y_.iloc[tr_idx],
            cat_features=cat_columns(f_tr),
            eval_set=(prepare_catboost(f_va), y_.iloc[va_idx]),
            early_stopping_rounds=50, verbose=False,
        )
        oof[va_idx] = model.predict_proba(prepare_catboost(f_va))[:, 1]
        te += model.predict_proba(prepare_catboost(f_te))[:, 1] / n_splits
        scores.append(roc_auc_score(y_.iloc[va_idx], oof[va_idx]))
    auc = roc_auc_score(y_, oof)
    print(f"[CAT     ] fold AUCs: " + "  ".join(f"{s:.5f}" for s in scores))
    print(f"[CAT     ] pooled OOF AUC: {auc:.5f}   ({time.time() - t0:.0f}s)")
    return oof, te

oof_cat, te_cat = cv_catboost(CAT_PARAMS, X, X_test, y)

## 8 · Model 3 — strong LightGBM V5 (the workhorse, weight 0.60)

Slower learning rate, more trees, finer splits (`max_bin=2047`), light
regularization. In the reference recipe this single model scores
≈ 0.9443 OOF — the best of the three — and is therefore weighted **0.60**
in the blend.

In [ ]:
LGB_STRONG = dict(
    n_estimators=1500, learning_rate=0.03, num_leaves=31, max_depth=-1,
    max_bin=2047, min_child_samples=30,
    subsample=0.85, colsample_bytree=0.9,
    reg_alpha=0.1, reg_lambda=0.2,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)

oof_lgb5, te_lgb5 = cv_lightgbm(LGB_STRONG, X, X_test, y, freq=True, tag="[LGB v5 ]")

## 9 · Ensemble & submission

Probability-blend the three fold-averaged test predictions:

| Model | Weight |
|---|---|
| LightGBM V4 (digits + frequency) | 0.20 |
| CatBoost | 0.20 |
| LightGBM V5 (strong) | 0.60 |

The weights come from the reference solution's OOF experiments (pooled OOF
AUC **0.944403**, public LB **0.94439**). Rank averaging and XGBoost were
tried there and only moved the score in the 5th decimal — probability
averaging with these weights is the compute/score sweet spot.

In [ ]:
WEIGHTS = {"lgb": 0.20, "cat": 0.20, "lgb5": 0.60}

oof_blend = WEIGHTS["lgb"] * oof_lgb + WEIGHTS["cat"] * oof_cat + WEIGHTS["lgb5"] * oof_lgb5
te_blend  = WEIGHTS["lgb"] * te_lgb  + WEIGHTS["cat"] * te_cat  + WEIGHTS["lgb5"] * te_lgb5

print("=== RESULTS ===")
print(f"LightGBM raw (baseline)  OOF AUC: {roc_auc_score(y, oof_base):.5f}")
print(f"LightGBM v4 (engineered) OOF AUC: {roc_auc_score(y, oof_lgb):.5f}")
print(f"CatBoost                 OOF AUC: {roc_auc_score(y, oof_cat):.5f}")
print(f"LightGBM v5 (strong)     OOF AUC: {roc_auc_score(y, oof_lgb5):.5f}")
print(f"ENSEMBLE (0.2/0.2/0.6)   OOF AUC: {roc_auc_score(y, oof_blend):.5f}")

assert te_blend.shape == (len(test),)
assert np.isfinite(te_blend).all() and ((te_blend >= 0) & (te_blend <= 1)).all()

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: te_blend})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved {SUBMISSION_PATH} ({submission.shape})")
submission.head()

## 10 · Submission sanity check

In [ ]:
assert list(submission.columns) == list(sample.columns) == [ID_COL, TARGET]
assert submission[ID_COL].equals(sample[ID_COL].reset_index(drop=True))  # same ids, same order
print("schema + id order match the sample submission  [ok]")
print()
print(submission[TARGET].describe().round(4))
print(f"\nexpected P(buy) on test: ~{y.mean():.3f}  |  submission mean: {submission[TARGET].mean():.4f}")

## 11 · Where to go from here

This notebook reproduces the top public-Leaderboard recipe. Levers, in rough
order of expected payoff (all from the public reference solutions):

1. **Multi-seed rank averaging** — re-run the ensemble with 3–5 seeds,
   convert each prediction set to ranks, average the ranks. Typically
   +0.0002–0.0003 AUC.
2. **Add XGBoost to the blend** — a 4th GBDT family member helped slightly
   in the reference repo's V2; re-tune the blend weights afterwards.
3. **Tune the strong LightGBM** with Optuna (`min_child_samples`, `max_bin`,
   `feature_fraction`) — ~20 trials, maximize pooled OOF AUC.
4. **Fold-safe target encoding** (with smoothing) for `City_Type` /
   `Current_Car_Type`; frequency-encoding other columns did **not** help —
   income is the only frequency that consistently improves.
5. **Pseudo-labeling** — score test rows, keep the confident tails, retrain
   once. Careful with the 17.5% base rate; validate on OOF first.
6. **Don't bother** (per ablations): K-Means cluster features, MLP/neural
   nets, most interaction terms — trees already capture
   `income × eco concern` and `commute ÷ chargers` themselves.

> **Key takeaway:** on synthetic Playground data, *data artifacts are
> signal*. The trailing digits of `Annual_Income_USD` are worth more than any
> conventional interaction term — always inspect a continuous feature at raw
> precision before engineering it.